# Semantic-SWIFT codec training experiments

This notebook trains and evaluates the semantic extension of the Swift layered
neural video codec. It supports the A/B/C ablation ladder over the three
temporal hierarchy levels.

The semantic code lives in `codec/semantic/`. The pristine baseline stays in
`codec/src/`. Each target frame gets two binary side-information layers:

- a SAM3 object mask `S` (semantic base layer),
- a Canny edge map `E` (structural layer).

The mask and edge maps steer the visual layer with a mask/edge-weighted loss
and, for ablation C, encoder/decoder conditioning (`MaskAdd` + `FiLM`). The
`S` and `E` bitstreams are lossless in v1 and have no trainable parameters;
the visual codec and the conditioning heads are trained.

## Outputs

Each run writes to `codec/semantic/experiments/hierarchy_<level>/<ABLATION>/`:

- `model/` - checkpoints,
- `output/` - packed codes and probe images,
- `tensorboard/` - TensorBoard events,
- `train.log` - console log.

Use `codec/semantic/semantic_inference.py` to decode per-layer videos and
per-layer PSNR/MS-SSIM for a trained checkpoint.

## Before you start

### 1. Select the environment

Run this command in a terminal:

```bash
conda activate semswift_env
```

### 2. Register the kernel

```bash
python -m ipykernel install --sys-prefix --name semswift_env --display-name "Python (semswift_env)"
python -m jupyter kernelspec list
```

### 3. Start JupyterLab

```bash
cd /nfs/home/olidiz18/git/S-SWIFT/codec/semantic
python -m jupyter lab
```

### 4. Select the kernel

Open the notebook and select `Kernel > Change Kernel > Python (semswift_env)`.

### 5. Run the training cells

Run the notebook cells from top to bottom. Run one training cell at a time.
Set `QUICK_TEST = True` in the configuration cell for a short check.


## Scope

- This notebook reproduces the semantic ablation ladder (A/B/C) for each of the
  three temporal-hierarchy levels.
- It runs `codec/semantic/train.py`. It does not run the adversarial variant.
- It reads the masks from `data/kinetics/train_masks` and the edges from
  `data/kinetics/train_edges` (and the `test_*` counterparts).
- Kinetics frames, masks, and edges are already 352x288, so the loader reads
  them directly (no resize).
- The mask/edge maps are coded losslessly. Their cost is measured in bytes per
  frame; mask IoU and edge F1 are trivially 1 and only recorded as placeholders.


In [13]:
from pathlib import Path
from datetime import datetime
import os
import signal
import subprocess
import sys

import cv2

PREFERRED_GPU_IDS = [0, 1]
AUTO_SELECT_GPUS = True
MIN_FREE_GPU_MEMORY_MB = 12288


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "codec" / "src" / "train.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot find the repository root. Start Jupyter in codec/semantic or in the repository root."
    )


def query_free_gpu_memory():
    command = [
        "nvidia-smi",
        "--query-gpu=index,memory.free",
        "--format=csv,noheader,nounits",
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    free_memory = {}
    for line in result.stdout.strip().splitlines():
        index, memory = [part.strip() for part in line.split(",")]
        free_memory[int(index)] = int(memory)
    return free_memory


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "codec" / "semantic"
DATA_DIR = REPO_ROOT / "data"

TRAIN_DIR = DATA_DIR / "kinetics" / "train"
EVAL_DIR = DATA_DIR / "kinetics" / "test"
TRAIN_MV_DIR = DATA_DIR / "kinetics" / "train_mv"
EVAL_MV_DIR = DATA_DIR / "kinetics" / "test_mv"
TRAIN_MASK_DIR = DATA_DIR / "kinetics" / "train_masks"
EVAL_MASK_DIR = DATA_DIR / "kinetics" / "test_masks"
TRAIN_EDGE_DIR = DATA_DIR / "kinetics" / "train_edges"
EVAL_EDGE_DIR = DATA_DIR / "kinetics" / "test_edges"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if AUTO_SELECT_GPUS:
    try:
        free_memory = query_free_gpu_memory()
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("Warning: Cannot read GPU memory from nvidia-smi.")
        free_memory = {
            index: MIN_FREE_GPU_MEMORY_MB for index in PREFERRED_GPU_IDS
        }

    SELECTED_PHYSICAL_GPU_IDS = [
        index for index in PREFERRED_GPU_IDS
        if free_memory.get(index, 0) >= MIN_FREE_GPU_MEMORY_MB
    ]
    if not SELECTED_PHYSICAL_GPU_IDS:
        print("Warning: No preferred GPU has enough free memory.")
        print("Free memory (MiB):", free_memory)
        SELECTED_PHYSICAL_GPU_IDS = list(PREFERRED_GPU_IDS)
else:
    SELECTED_PHYSICAL_GPU_IDS = list(PREFERRED_GPU_IDS)

os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(
    str(index) for index in SELECTED_PHYSICAL_GPU_IDS
)
GPU_IDS = list(range(len(SELECTED_PHYSICAL_GPU_IDS)))
GPU_ARG = ",".join(str(index) for index in GPU_IDS)

if "torch" in sys.modules:
    print("Warning: Restart the kernel before you rerun this cell. CUDA was already initialized.")

import torch

print("Kernel Python:", sys.executable)
if "semswift_env" not in sys.executable:
    print("Warning: This kernel does not use semswift_env. Select the Python (semswift_env) kernel.")

print("Repository root:", REPO_ROOT)
print("Source directory:", SRC_DIR)
print("Train data:", TRAIN_DIR)
print("Eval data:", EVAL_DIR)
print("Train masks:", TRAIN_MASK_DIR)
print("Train edges:", TRAIN_EDGE_DIR)
print("Physical GPU IDs:", SELECTED_PHYSICAL_GPU_IDS)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("Logical GPU IDs:", GPU_IDS)


Kernel Python: /nfs/home/olidiz18/conda_envs/semswift_env/bin/python
Repository root: /nfs/home/olidiz18/git/S-SWIFT
Source directory: /nfs/home/olidiz18/git/S-SWIFT/codec/semantic
Train data: /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train
Eval data: /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test
Train masks: /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_masks
Train edges: /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_edges
Physical GPU IDs: [0, 1]
CUDA_VISIBLE_DEVICES: 0,1
Logical GPU IDs: [0, 1]


## Configuration

`ABLATION` selects the experiment:

- `"A"` baseline (no mask/edge args),
- `"B"` weighted loss (`--alpha-sem`/`--alpha-edge`, conditioning off),
- `"C"` conditioning on top of B (`--use-conditioning`).

`QUICK_TEST` changes only the iteration counts. `NUM_WORKERS` sets the DataLoader
process count. `NUM_CROPS` sets how many crops each frame contributes.
`EVAL_BATCH_SIZE` sets the evaluation batch size. `RESUME_FROM` continues a
hierarchy run from a saved checkpoint (set a value to `None` to start fresh).


In [14]:
HIERARCHIES = {
    0: {"distance1": 6, "distance2": 6, "bits": 16, "encoder_fuse_level": 1, "decoder_fuse_level": 1},
    1: {"distance1": 3, "distance2": 3, "bits": 16, "encoder_fuse_level": 2, "decoder_fuse_level": 3},
    2: {"distance1": 1, "distance2": 2, "bits": 8, "encoder_fuse_level": 1, "decoder_fuse_level": 1},
}

ABLATION = "C"

ALPHA_SEM = 1.0
ALPHA_EDGE = 0.5
USE_CONDITIONING = True
USE_WLPIPS = True
LPIPS_WEIGHT = 0.1
USE_VMAF = False
USE_LPIPS = False

QUICK_TEST = False
MAX_TRAIN_ITERS = 20 if QUICK_TEST else 20000
CHECKPOINT_ITERS = 10 if QUICK_TEST else 5000
EVAL_ITERS = 10 if QUICK_TEST else 4000
SCHEDULE = ",".join(
    str(int(MAX_TRAIN_ITERS * fraction))
    for fraction in [0.5, 0.6, 0.7, 0.8, 0.9]
)

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
NUM_CROPS = 2
NUM_WORKERS = 8

EVAL_STRIDE = 8
EVAL_FP16 = False

RESUME_FROM = {0: None, 1: None, 2: None}
EARLY_STOP_PATIENCE = 0

SAVE_CODES = True
SAVE_OUT_IMG = True
TENSORBOARD = True
TENSORBOARD_PORT = 6006

EXPERIMENT_ROOT = SRC_DIR / "experiments"


def semantic_arguments(ablation=None):
    ablation = ABLATION if ablation is None else ablation
    args = []
    if ablation in ("B", "C"):
        args += [
            "--train-mask", str(TRAIN_MASK_DIR),
            "--eval-mask", str(EVAL_MASK_DIR),
            "--train-edge", str(TRAIN_EDGE_DIR),
            "--eval-edge", str(EVAL_EDGE_DIR),
            "--alpha-sem", str(ALPHA_SEM),
            "--alpha-edge", str(ALPHA_EDGE),
        ]
    if ablation == "C" and USE_CONDITIONING:
        args += ["--use-conditioning"]
    if USE_WLPIPS and ablation in ("B", "C"):
        args += ["--use-wlpips", "--lpips-weight", str(LPIPS_WEIGHT)]
    if USE_VMAF:
        args += ["--use-vmaf"]
    if USE_LPIPS:
        args += ["--use-lpips"]
    return args


def experiment_dir(level, ablation=None):
    ablation = ABLATION if ablation is None else ablation
    return EXPERIMENT_ROOT / f"hierarchy_{level}" / ablation


def model_dir(level, ablation=None):
    return experiment_dir(level, ablation) / "model"


def output_dir(level, ablation=None):
    return experiment_dir(level, ablation) / "output"


def log_path(level, ablation=None):
    return experiment_dir(level, ablation) / "train.log"


def tensorboard_dir(level, ablation=None):
    run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
    base = experiment_dir(level, ablation) / "tensorboard" / run_tag
    resume_iter = RESUME_FROM.get(level)
    if resume_iter:
        return base / f"resume_{resume_iter}"
    return base


def checkpoint_prefix(level):
    return f"hierarchy{level}"


for level in HIERARCHIES:
    model_dir(level).mkdir(parents=True, exist_ok=True)
    output_dir(level).mkdir(parents=True, exist_ok=True)

print("Experiment root:", EXPERIMENT_ROOT)
print("Ablation:", ABLATION)
print("Ablation directory pattern:", EXPERIMENT_ROOT / "hierarchy_<level>" / ABLATION)
print("Selected physical GPUs:", SELECTED_PHYSICAL_GPU_IDS)
print("Logical GPU IDs:", GPU_IDS)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("Semantic arguments:", semantic_arguments())
print("Save codes:", SAVE_CODES)
print("Save output images:", SAVE_OUT_IMG)
print("TensorBoard:", TENSORBOARD)
print("DataLoader workers:", NUM_WORKERS)
print("Num crops:", NUM_CROPS)
print("Eval batch size:", EVAL_BATCH_SIZE)
print("Resume from:", RESUME_FROM)

Experiment root: /nfs/home/olidiz18/git/S-SWIFT/codec/semantic/experiments
Ablation: C
Ablation directory pattern: /nfs/home/olidiz18/git/S-SWIFT/codec/semantic/experiments/hierarchy_<level>/C
Selected physical GPUs: [0, 1]
Logical GPU IDs: [0, 1]
CUDA_VISIBLE_DEVICES: 0,1
Semantic arguments: ['--train-mask', '/nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_masks', '--eval-mask', '/nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test_masks', '--train-edge', '/nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_edges', '--eval-edge', '/nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test_edges', '--alpha-sem', '1.0', '--alpha-edge', '0.5', '--use-conditioning', '--use-wlpips', '--lpips-weight', '0.1']
Save codes: True
Save output images: True
TensorBoard: True
DataLoader workers: 8
Num crops: 2
Eval batch size: 16
Resume from: {0: None, 1: None, 2: None}


## GPU check

The training script uses `torchrun` and `DistributedDataParallel`. The notebook
checks the GPU count before each run.


In [15]:
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
print("Physical GPU IDs:", SELECTED_PHYSICAL_GPU_IDS)
print("Logical GPU IDs:", GPU_IDS)

if torch.cuda.is_available():
    for logical in GPU_IDS:
        free, total = torch.cuda.mem_get_info(logical)
        print(
            f"Logical GPU {logical}: {torch.cuda.get_device_name(logical)} "
            f"free={free / 1024 ** 3:.1f} GiB total={total / 1024 ** 3:.1f} GiB"
        )
else:
    print(
        "Warning: CUDA is not available. Training will stop. "
        "Check the NVIDIA driver and the PyTorch CUDA build."
    )


def require_gpus():
    if len(GPU_IDS) == 0:
        raise RuntimeError(
            "No GPU is selected. Set PREFERRED_GPU_IDS or wait for a free GPU."
        )
    if not torch.cuda.is_available() or torch.cuda.device_count() < len(GPU_IDS):
        raise RuntimeError(
            f"Training needs {len(GPU_IDS)} visible GPUs. "
            f"Found {torch.cuda.device_count()}. "
            "Check the NVIDIA driver and the PyTorch CUDA build."
        )
    for logical in GPU_IDS:
        free, _ = torch.cuda.mem_get_info(logical)
        free_mb = free // (1024 ** 2)
        if free_mb < MIN_FREE_GPU_MEMORY_MB:
            raise RuntimeError(
                f"Logical GPU {logical} has {free_mb} MiB free. "
                f"Training needs at least {MIN_FREE_GPU_MEMORY_MB} MiB. "
                "Wait for a free GPU or change PREFERRED_GPU_IDS."
            )


PyTorch version: 2.14.1+cu126
PyTorch CUDA build: 12.6
CUDA available: True
Visible GPU count: 2
Physical GPU IDs: [0, 1]
Logical GPU IDs: [0, 1]
Logical GPU 0: NVIDIA RTX 4000 Ada Generation free=19.4 GiB total=19.5 GiB
Logical GPU 1: NVIDIA RTX 4000 Ada Generation free=19.4 GiB total=19.5 GiB


## Mask / edge inspection

Run this cell before any long training run. It loads one clip that has both a
mask and an edge map, overlays them on the frame, and confirms that a random
crop keeps the RGB, mask, and edge channels locked together. The dataset appends
`S` and `E` as the last two channels and applies `crop_cv2`/`flip_cv2` to the
whole tensor, so the semantic maps stay aligned with the patch. A horizontal
flip is a plain mirror and needs no sign change for binary maps.


In [16]:
import glob
import random
import numpy as np

from dataset import read_binary_map, crop_cv2

mask_clips = sorted({
    os.path.dirname(p) for p in glob.glob(str(TRAIN_MASK_DIR / "*" / "frame_*.png"))
})
edge_clips = sorted({
    os.path.dirname(p) for p in glob.glob(str(TRAIN_EDGE_DIR / "*" / "frame_*.png"))
})
common = sorted(set(mask_clips) & set(edge_clips))

if not common:
    print("No clip has both a mask and an edge map yet. The loader zero-fills missing maps.")
else:
    name = os.path.basename(common[0])
    frame_path = sorted((TRAIN_DIR / name).glob("frame_*.png"))[3]

    frame = cv2.cvtColor(cv2.imread(str(frame_path)), cv2.COLOR_BGR2RGB)
    H = (frame.shape[0] // 16) * 16
    W = (frame.shape[1] // 16) * 16
    frame = frame[:H, :W].astype(np.float32)

    mask = read_binary_map(str(TRAIN_MASK_DIR / name / frame_path.name), H, W)[..., 0]
    edge = read_binary_map(str(TRAIN_EDGE_DIR / name / frame_path.name), H, W)[..., 0]

    print("Clip:", name)
    print("Frame:", frame_path.name, "shape:", (H, W))
    print("Mask foreground fraction: %.4f" % mask.mean())
    print("Edge foreground fraction: %.4f" % edge.mean())

    overlay = frame.copy()
    overlay[mask > 0.5] *= 0.4
    overlay[mask > 0.5, 0] += 255.0 * 0.6
    overlay[edge > 0.5] *= 0.4
    overlay[edge > 0.5, 1] += 255.0 * 0.6
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    sample = np.concatenate(
        [frame, mask[..., None] * 255.0, edge[..., None] * 255.0], axis=2
    )
    random.seed(0)
    crop = crop_cv2(sample, 64)
    assert crop.shape[2] == 5, crop.shape
    print("Crop shape:", crop.shape)
    print("Mask crop unique:", np.unique(crop[..., 3]).tolist())
    print("Edge crop unique:", np.unique(crop[..., 4]).tolist())

    (EXPERIMENT_ROOT).mkdir(parents=True, exist_ok=True)
    out_path = EXPERIMENT_ROOT / "mask_edge_inspection.png"
    cv2.imwrite(str(out_path), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    print("Overlay written to:", out_path)


No clip has both a mask and an edge map yet. The loader zero-fills missing maps.


## Check the data

The next cell builds the dataset objects for each hierarchy level. The count
depends on the hierarchy positions and on complete frame groups.


In [17]:
from dataset import ImageFolder
from train_options import parser


def base_arguments(level, ablation=None):
    config = HIERARCHIES[level]
    return [
        "--train", str(TRAIN_DIR),
        "--eval", str(EVAL_DIR),
        "--train-mv", str(TRAIN_MV_DIR),
        "--eval-mv", str(EVAL_MV_DIR),
        *semantic_arguments(ablation),
        "--encoder-fuse-level", str(config["encoder_fuse_level"]),
        "--decoder-fuse-level", str(config["decoder_fuse_level"]),
        "--v-compress", "--warp", "--stack", "--fuse-encoder",
        "--bits", str(config["bits"]),
        "--distance1", str(config["distance1"]),
        "--distance2", str(config["distance2"]),
        "--num-workers", str(NUM_WORKERS),
    ]


def dataset_arguments(level, ablation=None):
    return parser.parse_args(
        base_arguments(level, ablation)
        + [
            "--batch-size", str(BATCH_SIZE),
            "--eval-batch-size", str(EVAL_BATCH_SIZE),
            "--num-crops", str(NUM_CROPS),
            "--gpus", GPU_ARG,
        ]
    )


for level in HIERARCHIES:
    arguments = dataset_arguments(level)
    train_dataset = ImageFolder(
        is_train=True,
        root=str(TRAIN_DIR),
        mv_dir=str(TRAIN_MV_DIR),
        args=arguments,
    )
    eval_dataset = ImageFolder(
        is_train=False,
        root=str(EVAL_DIR),
        mv_dir=str(EVAL_MV_DIR),
        args=arguments,
    )
    print(
        f"Hierarchy {level}: {len(train_dataset)} training frames, "
        f"{len(eval_dataset)} evaluation frames."
    )

64000 images loaded.
	distance=6/6
16000 images loaded.
	distance=6/6
Hierarchy 0: 64000 training frames, 16000 evaluation frames.
128000 images loaded.
	distance=3/3
32000 images loaded.
	distance=3/3
Hierarchy 1: 128000 training frames, 32000 evaluation frames.
528000 images loaded.
	distance=1/2
132000 images loaded.
	distance=1/2
Hierarchy 2: 528000 training frames, 132000 evaluation frames.


## Multi-GPU training

Each training cell starts `codec/semantic/train.py` with `torchrun` and
`DistributedDataParallel`. Each ablation writes to
`codec/semantic/experiments/hierarchy_<level>/<ABLATION>/train.log`. Run one
cell at a time.

In [18]:
def training_command(level, ablation=None):
    command = [
        sys.executable, "-m", "torch.distributed.run",
        "--nproc_per_node", str(len(GPU_IDS)),
        str(SRC_DIR / "train.py"),
        *base_arguments(level, ablation),
        "--max-train-iters", str(MAX_TRAIN_ITERS),
        "--checkpoint-iters", str(CHECKPOINT_ITERS),
        "--eval-iters", str(EVAL_ITERS),
        "--model-dir", str(model_dir(level, ablation)),
        "--out-dir", str(output_dir(level, ablation)),
        "--save-model-name", checkpoint_prefix(level),
        "--batch-size", str(BATCH_SIZE),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--num-crops", str(NUM_CROPS),
        "--mp-context", "spawn",
        "--schedule", SCHEDULE,
        "--eval-stride", str(EVAL_STRIDE),
    ]
    resume_iter = RESUME_FROM.get(level)
    if resume_iter:
        command.extend([
            "--load-model-name", checkpoint_prefix(level),
            "--load-iter", str(resume_iter),
        ])
    if EARLY_STOP_PATIENCE > 0:
        command.extend(["--early-stop", str(EARLY_STOP_PATIENCE)])
    if EVAL_FP16:
        command.append("--eval-fp16")
    if SAVE_CODES:
        command.append("--save-codes")
    if SAVE_OUT_IMG:
        command.append("--save-out-img")
    if TENSORBOARD:
        command.extend(["--tensorboard-dir", str(tensorboard_dir(level, ablation))])
    return command


def run_training(level, ablation=None):
    ablation = ABLATION if ablation is None else ablation
    require_gpus()
    command = training_command(level, ablation)
    log_file = log_path(level, ablation)
    log_file.parent.mkdir(parents=True, exist_ok=True)

    print("Command:")
    print(" ".join(str(part) for part in command))
    print("Ablation:", ablation)
    print("Physical GPU IDs:", SELECTED_PHYSICAL_GPU_IDS)
    print("Logical GPU IDs:", GPU_ARG)
    print("Log file:", log_file)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        command,
        cwd=SRC_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if log_file.exists():
        backup = log_file.with_name(
            f"train_{datetime.now():%Y%m%d_%H%M%S}.log"
        )
        log_file.replace(backup)

    try:
        with log_file.open("w") as log:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise

    if return_code != 0:
        if return_code < 0:
            signal_number = -return_code
            try:
                signal_name = signal.Signals(signal_number).name
            except ValueError:
                signal_name = str(signal_number)
            raise RuntimeError(
                f"Training was killed by signal {signal_name} ({return_code}). "
                f"Read {log_file}. SIGKILL usually means the OS out-of-memory "
                "killer or an administrator stopped the process."
            )
        raise RuntimeError(f"Training stopped with code {return_code}. Read {log_file}.")

    print("Training finished.")

## TensorBoard

The training script writes TensorBoard events to
`codec/semantic/experiments/hierarchy_<level>/<ABLATION>/tensorboard/`.

If the notebook magic does not start, run this command in a terminal:

```bash
cd /nfs/home/olidiz18/git/S-SWIFT
tensorboard --logdir codec/semantic/experiments --port 6006 --bind_all
```

In [19]:
import os
import signal
import socket
import subprocess
import tempfile
import time


def _tensorboard_child_pids(pid):
    try:
        output = subprocess.run(
            ["ps", "-eo", "pid=,ppid="],
            capture_output=True, text=True, check=False,
        ).stdout
    except OSError:
        return []
    children = {}
    for line in output.splitlines():
        parts = line.split()
        if len(parts) != 2:
            continue
        try:
            child, parent = (int(part) for part in parts)
        except ValueError:
            continue
        children.setdefault(parent, []).append(child)
    result = []
    stack = [pid]
    while stack:
        for child in children.get(stack.pop(), []):
            result.append(child)
            stack.append(child)
    return result


def _tensorboard_pid_alive(pid):
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    return True


def _tensorboard_process_matches(pid, port):
    try:
        with open(f"/proc/{pid}/cmdline", "rb") as cmdline:
            parts = [
                part for part in
                cmdline.read().decode("utf-8", "replace").split("\x00")
                if part
            ]
    except OSError:
        return False
    if not any("tensorboard" in part for part in parts):
        return False
    if f"--port={port}" in parts:
        return True
    if "--port" in parts:
        return str(port) in parts
    return True


def _tensorboard_pids_on_port(port):
    pids = set()
    for command in (
        ["lsof", "-ti", f"tcp:{port}", "-sTCP:LISTEN"],
        ["fuser", f"{port}/tcp"],
    ):
        try:
            completed = subprocess.run(
                command, capture_output=True, text=True, check=False,
            )
        except OSError:
            continue
        for part in completed.stdout.split():
            try:
                pids.add(int(part))
            except ValueError:
                pass
        if pids:
            break
    return pids


def _tensorboard_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.2)
        return sock.connect_ex(("127.0.0.1", port)) == 0


def stop_tensorboard(port, timeout=8.0):
    pids = _tensorboard_pids_on_port(port)
    try:
        from tensorboard import manager as tensorboard_manager
        infos = tensorboard_manager.get_all()
    except Exception:
        tensorboard_manager = None
        infos = []
    if tensorboard_manager is not None:
        info_dir = os.path.join(tempfile.gettempdir(), ".tensorboard-info")
        for info in infos:
            if info.port != port:
                continue
            try:
                os.unlink(os.path.join(info_dir, f"pid-{info.pid}.info"))
            except OSError:
                pass
            if info.pid > 1 and info.pid != os.getpid():
                pids.add(info.pid)

    targets = set()
    for pid in pids:
        if pid <= 1 or pid == os.getpid():
            continue
        if not _tensorboard_process_matches(pid, port):
            continue
        if _tensorboard_pid_alive(pid):
            targets.add(pid)
            targets.update(_tensorboard_child_pids(pid))
    for pid in sorted(targets):
        print(f"Stopping TensorBoard (pid {pid})")
        try:
            os.kill(pid, signal.SIGTERM)
        except (ProcessLookupError, PermissionError):
            pass

    deadline = time.time() + timeout
    while time.time() < deadline and _tensorboard_port_open(port):
        time.sleep(0.1)

    if _tensorboard_port_open(port):
        for pid in sorted(targets):
            try:
                os.kill(pid, signal.SIGKILL)
            except (ProcessLookupError, PermissionError):
                pass
        print(f"Warning: port {port} is still in use")


try:
    ipython = get_ipython()
except NameError:
    ipython = None

if ipython is None:
    print("Start TensorBoard in a terminal:")
    print(f"tensorboard --logdir {EXPERIMENT_ROOT} --port {TENSORBOARD_PORT} --bind_all")
else:
    stop_tensorboard(TENSORBOARD_PORT)
    ipython.run_line_magic("load_ext", "tensorboard")
    ipython.run_line_magic(
        "tensorboard",
        f"--logdir {EXPERIMENT_ROOT} --port {TENSORBOARD_PORT}",
    )


Stopping TensorBoard (pid 8848)
Stopping TensorBoard (pid 8872)
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## Ablations

The codec-level experiment compares three configurations against the baseline in
`codec/src`:

- **A — baseline**: the copied architecture, no semantic args.
- **B — + weighted loss**: `--alpha-sem`/`--alpha-edge` set, conditioning off.
  Isolates the loss contribution.
- **C — + conditioning**: `--use-conditioning` on top of B. Isolates the
  conditioning contribution.

Set `ABLATION` and run the matching cell. Each ablation writes to its own
`experiments/hierarchy_<level>/<ABLATION>/` subtree, so A/B/C runs do not
overwrite each other.

### Ablation A — baseline

No mask/edge arguments. The loss and the architecture are unchanged from the
copied baseline. `W` degrades to all-ones, so this reproduces the original
training in the semantic tree.


In [20]:
ABLATION = "A"
USE_CONDITIONING = False
run_training(0)


Command:
/nfs/home/olidiz18/conda_envs/semswift_env/bin/python -m torch.distributed.run --nproc_per_node 2 /nfs/home/olidiz18/git/S-SWIFT/codec/semantic/train.py --train /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train --eval /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test --train-mv /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_mv --eval-mv /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test_mv --encoder-fuse-level 1 --decoder-fuse-level 1 --v-compress --warp --stack --fuse-encoder --bits 16 --distance1 6 --distance2 6 --num-workers 8 --max-train-iters 20000 --checkpoint-iters 5000 --eval-iters 4000 --model-dir /nfs/home/olidiz18/git/S-SWIFT/codec/semantic/experiments/hierarchy_0/A/model --out-dir /nfs/home/olidiz18/git/S-SWIFT/codec/semantic/experiments/hierarchy_0/A/output --save-model-name hierarchy0 --batch-size 8 --eval-batch-size 16 --num-crops 2 --mp-context spawn --schedule 10000,12000,14000,16000,18000 --eval-stride 8 --save-codes --save-out-img --tensorboard-dir /nfs/

### Ablation B — + weighted loss

Adds `--train-mask`/`--eval-mask`, `--train-edge`/`--eval-edge` and
`--alpha-sem`/`--alpha-edge`. The loss becomes
`W = 1 + alpha_sem * S + alpha_edge * E` (mean-normalized per sample), weighting
every reconstruction term toward the mask and edge regions. No architecture
change, so this stays checkpoint-compatible with the baseline.


In [ ]:
ABLATION = "B"
USE_CONDITIONING = False
run_training(0)


### Ablation C — + conditioning

Adds `--use-conditioning` on top of B: an additive `MaskAdd` branch in the
encoder and FiLM modulation in `DecoderCell`/`DecoderCell2` at the four
pixel-shuffle scales. Only the new heads are randomly initialized; the existing
conv/ConvLSTM weights still load from a baseline checkpoint.


In [21]:
ABLATION = "C"
USE_CONDITIONING = True
run_training(0)


Command:
/nfs/home/olidiz18/conda_envs/semswift_env/bin/python -m torch.distributed.run --nproc_per_node 2 /nfs/home/olidiz18/git/S-SWIFT/codec/semantic/train.py --train /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train --eval /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test --train-mv /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_mv --eval-mv /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test_mv --train-mask /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_masks --eval-mask /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test_masks --train-edge /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/train_edges --eval-edge /nfs/home/olidiz18/git/S-SWIFT/data/kinetics/test_edges --alpha-sem 1.0 --alpha-edge 0.5 --use-conditioning --use-wlpips --lpips-weight 0.1 --encoder-fuse-level 1 --decoder-fuse-level 1 --v-compress --warp --stack --fuse-encoder --bits 16 --distance1 6 --distance2 6 --num-workers 8 --max-train-iters 20000 --checkpoint-iters 5000 --eval-iters 4000 --model-dir /nfs/home/oli

### Train all hierarchy levels

Once the ablation is chosen, run the three hierarchy levels in sequence. Each
level writes its own model under
`experiments/hierarchy_<level>/<ABLATION>/`.

In [ ]:
for level in sorted(HIERARCHIES):
    run_training(level)


### Train all ablations and all hierarchies

Run the full A/B/C x hierarchy grid in one go. Each combination writes to
its own `experiments/hierarchy_<level>/<ABLATION>/` folder, so no manual
moving is needed. This is long: roughly 3 hours per run (9 runs total).


In [ ]:
for ablation in ("A", "B", "C"):
    ABLATION = ablation
    USE_CONDITIONING = (ablation == "C")
    for level in sorted(HIERARCHIES):
        run_training(level, ablation)


## Evaluation

The training script evaluates the model during training. The next cells load a
saved checkpoint and run evaluation again. The evaluation also reports the mask
and edge bitrate in bytes per frame (and VMAF/LPIPS when those flags are set).


In [ ]:
from dataset import get_loader
from evaluate import run_eval
from util import get_models, set_eval

import network

CHECKPOINT_NAMES = ["encoder", "binarizer", "decoder", "d2", "unet"]


def evaluation_arguments(level, ablation=None):
    arguments = base_arguments(level, ablation) + [
        "--model-dir", str(model_dir(level, ablation)),
        "--out-dir", str(output_dir(level, ablation)),
        "--save-model-name", checkpoint_prefix(level),
        "--batch-size", str(BATCH_SIZE),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--num-crops", str(NUM_CROPS),
        "--gpus", GPU_ARG,
    ]
    if SAVE_CODES:
        arguments.append("--save-codes")
    if SAVE_OUT_IMG:
        arguments.append("--save-out-img")
    return parser.parse_args(arguments)


def build_evaluation_model(level, ablation=None):
    args = evaluation_arguments(level, ablation)
    encoder, binarizer, decoder, unet = get_models(
        args=args,
        v_compress=args.v_compress,
        bits=args.bits,
        encoder_fuse_level=args.encoder_fuse_level,
        decoder_fuse_level=args.decoder_fuse_level,
    )
    d2 = network.DecoderCell2(
        v_compress=args.v_compress,
        shrink=args.shrink,
        bits=args.bits,
        fuse_level=args.decoder_fuse_level,
        use_conditioning=args.use_conditioning,
    ).cuda()

    nets = [encoder, binarizer, decoder, d2]
    if unet is not None:
        nets.append(unet)
    return args, nets


def latest_iteration(level, ablation=None):
    prefix = checkpoint_prefix(level)
    model_path = model_dir(level, ablation)
    paths = [
        p for p in model_path.glob(f"{prefix}_encoder_*.pth")
        if p.stem.rsplit("_", 1)[-1].isdigit()
    ]
    if not paths:
        raise FileNotFoundError(f"No checkpoint found in {model_path}")
    return int(paths[-1].stem.rsplit("_", 1)[-1])


def load_checkpoint(level, iteration, nets, ablation=None):
    prefix = checkpoint_prefix(level)
    model_path = model_dir(level, ablation)
    for name, net in zip(CHECKPOINT_NAMES, nets):
        path = model_path / f"{prefix}_{name}_{iteration:08d}.pth"
        if not path.exists():
            raise FileNotFoundError(path)
        state = torch.load(path, map_location="cuda:0")
        try:
            net.load_state_dict(state)
        except RuntimeError:
            state = {key.replace("module.", ""): value for key, value in state.items()}
            net.load_state_dict(state)


def evaluate_checkpoint(level, iteration=None, ablation=None):
    ablation = ABLATION if ablation is None else ablation
    require_gpus()
    if iteration is None:
        iteration = latest_iteration(level, ablation)

    args, nets = build_evaluation_model(level, ablation)
    load_checkpoint(level, iteration, nets, ablation)
    set_eval(nets)

    loader = get_loader(False, str(EVAL_DIR), str(EVAL_MV_DIR), args)
    result = run_eval(
        nets,
        loader,
        args,
        output_suffix=f"hierarchy{level}_iter{iteration}",
    )
    (loss, msssim, psnr, psnr_ee1, psnr_ee2, psnr_ee3, psnr_ee4,
     mask_bitrate, edge_bitrate, vmaf_mean, lpips_mean) = result

    print(f"Ablation {ablation}, Hierarchy {level}, iteration {iteration}")
    print(
        f"{'Step':>4} {'Loss':>10} {'MS-SSIM':>10} {'PSNR':>10} "
        f"{'EE1':>10} {'EE2':>10} {'EE3':>10} {'EE4':>10}"
    )
    for step in range(len(loss)):
        print(
            f"{step + 1:>4} {loss[step]:>10.5f} {msssim[step]:>10.5f} "
            f"{psnr[step]:>10.5f} {psnr_ee1[step]:>10.5f} "
            f"{psnr_ee2[step]:>10.5f} {psnr_ee3[step]:>10.5f} "
            f"{psnr_ee4[step]:>10.5f}"
        )
    if mask_bitrate is not None:
        print("Mask bytes/frame: %.2f" % mask_bitrate)
    if edge_bitrate is not None:
        print("Edge bytes/frame: %.2f" % edge_bitrate)
    if vmaf_mean is not None:
        print("VMAF: %.4f" % vmaf_mean)
    if lpips_mean is not None:
        print("LPIPS: %.4f" % lpips_mean)

    return {
        "loss": loss,
        "msssim": msssim,
        "psnr": psnr,
        "psnr_ee1": psnr_ee1,
        "psnr_ee2": psnr_ee2,
        "psnr_ee3": psnr_ee3,
        "psnr_ee4": psnr_ee4,
        "mask_bitrate": mask_bitrate,
        "edge_bitrate": edge_bitrate,
        "vmaf": vmaf_mean,
        "lpips": lpips_mean,
    }

### Run evaluation

Set `EVAL_ABLATION` to the ablation (`A`, `B`, or `C`), `EVAL_LEVEL` to a
hierarchy level, and `EVAL_ITERATION` to a checkpoint iteration (`None` for the
latest).

In [ ]:
EVAL_ABLATION = "C"
EVAL_LEVEL = 2
EVAL_ITERATION = None

results = evaluate_checkpoint(EVAL_LEVEL, EVAL_ITERATION, EVAL_ABLATION)

## Artifacts

- Models: `codec/semantic/experiments/hierarchy_<level>/<ABLATION>/model/`
- Logs: `codec/semantic/experiments/hierarchy_<level>/<ABLATION>/train.log`
- TensorBoard events: `codec/semantic/experiments/hierarchy_<level>/<ABLATION>/tensorboard/`
- Codes: `codec/semantic/experiments/hierarchy_<level>/<ABLATION>/output/iter<iteration>/codes/`
- Probe images: `codec/semantic/experiments/hierarchy_<level>/<ABLATION>/output/probe/`

Do not commit the experiment outputs.

## Notes

- The mask and edge bitstreams are lossless, so mask IoU and edge F1 are
  trivially 1. The operative metric for layers 0/1 is bytes per frame.
- `W = 1 + alpha_sem * S + alpha_edge * E` is normalized to mean 1 per sample,
  so all-zero masks degrade gracefully to the baseline loss.
- The semantic conditioning heads are randomly initialized; the existing
  conv/ConvLSTM weights still load from a baseline checkpoint.
- `NUM_CROPS = 4` amortizes the U-Net but uses more GPU memory. The default is 2.
- The loader zero-fills missing mask/edge files, so training is safe while masks
  are still generating.
- `DistributedDataParallel` runs one process per selected GPU.

## Troubleshooting

- If `torch.cuda.is_available()` is `False`, check the NVIDIA driver and the
  PyTorch CUDA build for `semswift_env`.
- If a GPU has less than 12 GiB free, wait for it or set `PREFERRED_GPU_IDS`.
- If the data check shows zero frames, check the clip folders under
  `data/kinetics/`.
- If a training cell stops, read the log file for the last error.
- If training stops with code -9, the OS killed the process. Check host memory
  and set `RESUME_FROM` to the last checkpoint.
- `--use-wlpips` needs a torchvision VGG feature network; leave it off if the
  dependency is missing.

## Source compatibility

The semantic tree is a copy of `codec/src/` plus semantic extensions. The two
trees will diverge; re-apply baseline fixes manually. The semantic changes are:

- `dataset.py` appends the target frame's mask and edge as channels 13-14.
- `util.py` threads `S`/`E` through `prepare_batch` and `forward_model`.
- `train.py` uses the weighted loss and the conditioning branches.
- `network.py` adds `MaskAdd` (encoder) and `FiLM` (decoders) behind
  `--use-conditioning`.
- `semantic_loss.py`, `conditioning.py`, `mask_codec.py`, and
  `semantic_metrics.py` are new modules.
